# 🎬 AI Visual Novel Engine - Colab Host
This notebook runs **Ollama (for LLM dialogue/narration)** and **ComfyUI (for visual scene generation)** on Google Colab's GPU and exposes them via secure Cloudflare tunnels. 

### 🚀 How to use:
1. In the menu bar, go to **Runtime** > **Change runtime type** and ensure **GPU T4** (or higher) is selected.
2. Run **Step 1** to install Cloudflare Tunneling tools.
3. Run **Step 2** to download and start Ollama.
4. Run **Step 3** to set up ComfyUI and download a base Stable Diffusion model.
5. Copy the generated `.trycloudflare.com` URLs and paste them into your desktop app's **AI Settings** panel!

--- 
## 🛠️ Step 1: Install Tunnel Tooling (Cloudflare Tunnel)

In [ ]:
# Download cloudflared binary to expose local ports to public HTTPS URLs
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("[+] Cloudflared tunnel client installed successfully!")

--- 
## 🧠 Step 2: Set up Ollama (Text Generation)
Runs the LLM model on Colab's GPU. You can change `MODEL_NAME` to any model supported by Ollama.

In [ ]:
MODEL_NAME = "llama3:8b"  # @param ["llama3:8b", "mistral", "phi3", "gemma:7b"]

import os
import subprocess
import time
import re

print("[-] Downloading and installing Ollama...")
# Install zstd dependency required for extraction
!apt-get update -qq && apt-get install -y -qq zstd
# Run the official Linux installer script
!curl -fsSL https://ollama.com/install.sh | sh

print("[-] Starting Ollama server in background...")
# Spawn using absolute path and custom environmental origins to prevent 403 Forbidden tunnel rejections
env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_HOST"] = "0.0.0.0"
ollama_server = subprocess.Popen(["/usr/local/bin/ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

print(f"[-] Pulling model '{MODEL_NAME}' (this might take a few minutes)...")
!/usr/local/bin/ollama pull {MODEL_NAME}

print("[-] Exposing Ollama (Port 11434) via Cloudflare Tunnel...")
with open("ollama_tunnel.log", "w") as log:
    # Rewrite Host header to localhost:11434 to prevent Ollama from rejecting requests with 403 Forbidden
    subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434"], stdout=log, stderr=log)

print("[-] Waiting for tunnel public URL...")
time.sleep(8)
url = None
if os.path.exists("ollama_tunnel.log"):
    with open("ollama_tunnel.log", "r") as log:
        for line in log:
            match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
            if match:
                url = match.group(0)
                break

if url:
    print("\n=========================================================")
    print(f"🔥 OLLAMA PUBLIC API URL: {url}")
    print(f"   Paste this into 'LLM API URL' and set model as '{MODEL_NAME}'")
    print("=========================================================\n")
else:
    print("[!] Failed to automatically parse tunnel URL. Please check 'ollama_tunnel.log' to extract the trycloudflare.com link manually.")

--- 
## 🎨 Step 3: Set up ComfyUI (Image Generation)
Installs ComfyUI, downloads a checkpoint, and hosts the visual renderer.

In [ ]:
print("[-] Cloning ComfyUI repository...")
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI

print("[-] Installing python requirements...")
!pip install -q -r requirements.txt

print("[-] Downloading SD 1.5 Base Checkpoint (DreamShaper)...")
# Download a fast, stylized model to models/checkpoints
!wget -q -O models/checkpoints/dreamshaper_8.safetensors https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors

print("[-] Starting ComfyUI server and creating Tunnel...")
import subprocess
import time
import os
import re

# Start ComfyUI in the background
comfy_server = subprocess.Popen(["python", "main.py", "--port", "8188", "--listen"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Expose port 8188
with open("comfy_tunnel.log", "w") as log:
    subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8188"], stdout=log, stderr=log)

print("[-] Waiting for tunnel public URL...")
time.sleep(8)
url = None
if os.path.exists("comfy_tunnel.log"):
    with open("comfy_tunnel.log", "r") as log:
        for line in log:
            match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
            if match:
                url = match.group(0)
                break

if url:
    print("\n=========================================================")
    print(f"🎨 COMFYUI PUBLIC API URL: {url}")
    print("   Paste this into 'ComfyUI Endpoint URL' in your App Settings")
    print("=========================================================\n")
else:
    print("[!] Failed to automatically parse tunnel URL. Please check 'comfy_tunnel.log' to extract the trycloudflare.com link manually.")